# ハンズオン① LLaMA 3 + LoRA + SFTTrainer による SFT

**対応セクション**: 2-1「SFT (Supervised Fine-Tuning) + LoRA」  
**推奨所要時間**: 約 90 分  
**必要環境**: GPU (A100 推奨)、HF_HOME 設定済み

---

## このノートブックの目標

1. LLaMA 3 8B に 4bit 量子化（QLoRA）を適用してメモリを節約する
2. LoRA アダプタを設定し、学習パラメータ数を確認する
3. 日本語指示データセット（dolly-15k-ja）で SFT を実行する
4. wandb でトレーニングログを監視する
5. LoRA の rank を変えて学習コストと出力品質のトレードオフを観察する

> **前提**: GPU セッションを占有します。実行前に `/data/shared/gpu_queue.txt` で順番を確認してください。  
> `tmux new -s sft-train` セッション内で実行することを推奨します。

## 0. セットアップ確認

In [ ]:
# 実行時間: 数秒
import os
import torch

# HF キャッシュを共有ストレージに向ける
os.environ.setdefault('HF_HOME', '/data/shared/hf_cache')
os.environ.setdefault('TRANSFORMERS_CACHE', '/data/shared/hf_cache')

print('=== 環境確認 ===')
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM            : {total_vram:.1f} GB')
    print(f'HF_HOME         : {os.environ["HF_HOME"]}')
else:
    print('⚠️  GPU が検出されません。このノートブックの SFT 学習は GPU が必要です。')

---

## Step 1: ライブラリのインポート

In [ ]:
# 実行時間: 数秒
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

print('ライブラリのインポート完了')

---

## Step 2: モデルの読み込み（4bit QLoRA）

LLaMA 3 8B をそのまま読み込むと約 16GB の VRAM が必要です。  
4bit 量子化（NF4）を使うと約 5GB に削減できます。

In [ ]:
# 実行時間: 約2〜3分（初回はダウンロードあり）
MODEL_NAME = 'meta-llama/Meta-Llama-3-8B'

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 4bit 量子化設定
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print('モデルを読み込み中...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config if device == 'cuda' else None,
    device_map='auto' if device == 'cuda' else None,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('\nモデル読み込み完了')
if device == 'cuda':
    used = torch.cuda.memory_allocated() / 1e9
    print(f'使用 VRAM: {used:.1f} GB')

---

## Step 3: LoRA アダプタの設定

元のモデルパラメータは凍結したまま、差分行列 $BA$ だけを学習します。  
rank を変えると学習パラメータ数がどう変わるか確認しましょう。

In [ ]:
# 実行時間: 数秒
# ── ここを変えて実験しよう ──────────────────────────────────
LORA_RANK  = 16    # 試す値: 8, 16, 32, 64
LORA_ALPHA = 32    # 通常は rank * 2
# ──────────────────────────────────────────────────────────

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    bias='none',
)

model = get_peft_model(model, lora_config)

# 学習可能パラメータ数を確認
model.print_trainable_parameters()

---

## Step 4: データセットの準備

日本語指示データ `kunishou/databricks-dolly-15k-ja` を使用します。  
各サンプルを Alpaca 形式のプロンプトに変換します。

In [ ]:
# 実行時間: 約1〜2分（初回はダウンロードあり）
def format_instruction(sample: dict) -> str:
    """Alpaca 形式のプロンプトテンプレート"""
    instruction = sample.get('instruction', '')
    context     = sample.get('context', '')
    response    = sample.get('response', '')

    if context:
        return (
            f'### 指示:\n{instruction}\n\n'
            f'### 文脈:\n{context}\n\n'
            f'### 回答:\n{response}'
        )
    return f'### 指示:\n{instruction}\n\n### 回答:\n{response}'


dataset = load_dataset('kunishou/databricks-dolly-15k-ja', split='train')
dataset = dataset.map(
    lambda x: {'text': format_instruction(x)},
    remove_columns=dataset.column_names,
)

print(f'データ件数: {len(dataset)}')
print('\n--- サンプル（最初の1件）---')
print(dataset[0]['text'][:400])

---

## Step 5: 学習の実行

デモ用に `max_steps=50` で短縮しています。  
本格的な学習は `scripts/train_sft.py` を tmux セッションで実行してください。

In [ ]:
# 実行時間: 約5〜10分（max_steps=50 の場合）
from datetime import datetime

OUTPUT_DIR = f'./outputs/sft_notebook_{datetime.now().strftime("%H%M%S")}'

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_steps=50,                    # デモ用に短縮
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    bf16=device == 'cuda',
    logging_steps=10,
    save_steps=50,
    report_to='wandb',
    run_name=f'sft-rank{LORA_RANK}-demo',
    max_seq_length=512,
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print('学習開始...')
trainer.train()
print('学習完了！')

---

## Step 6: 学習後のモデルで推論

SFT 前後で回答がどう変わるか比較します。

In [ ]:
# 実行時間: 約30秒
model.eval()

prompt = '### 指示:\n機械学習と深層学習の違いを100字以内で説明してください。\n\n### 回答:\n'

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('=== SFT 後の回答 ===')
print(generated)

---

## 実験: rank を変えてみよう

Step 3 の `LORA_RANK` を以下の値で実験し、結果を比較してください。

| rank | 学習可能パラメータ | 期待される変化 |
|---|---|---|
| 8 | 最少 | 学習が速い・精度は低め |
| 16 | バランス | 推奨スタート地点 |
| 32 | 多め | より複雑なタスクに対応 |
| 64 | 多い | ほぼフルFTに近い挙動 |

---

## まとめ

1. **QLoRA** で LLaMA 3 8B を 4bit 量子化し、A100 40GB の VRAM に収める
2. **LoRA** は差分行列 $\Delta W = BA$ のみを学習し、全パラメータの 0.1〜0.5% だけを更新する
3. **SFTTrainer** で指示データセットを使って「指示に従う能力」を付与する
4. **rank** が大きいほど表現力が増すが、メモリと計算コストが増加する

次のハンズオンでは、DPO で応答スタイルの preference alignment に挑戦します。